## Muon_skript_reco_test

In [ ]:
import sqlite3
import pandas as pd

db_path = "/lustre/hpc/project/icecube/MonteCarlo2022/databases/Muon/Muon_merged.db"
#db_path = "/groups/icecube/janikh/PREP/I3_read_out/Data_storage/merged/events.db"
# mode=ro: read-only
# immutable=1: SQLite versucht keine Locks/Journale anzulegen (sehr hilfreich auf HPC/readonly)
uri = f"file:{db_path}?mode=ro&immutable=1"

conn = sqlite3.connect(uri, uri=True)

# optional: extra Safety – verhindert Writes auch innerhalb der Session
conn.execute("PRAGMA query_only = ON;")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

tables.head()

In [ ]:
pd.read_sql("PRAGMA table_info(SplitInIcePulses);", conn)

In [ ]:
pd.read_sql("PRAGMA table_info(truth);", conn)

In [ ]:
from tqdm.notebook import tqdm

# Anzahl der Events in jeder Tabelle zählen (mit Ladebalken)
tables_to_count = {
    "truth": "SELECT COUNT(DISTINCT event_no) AS n_events FROM truth;",
    "SplitInIcePulses": "SELECT COUNT(DISTINCT event_no) AS n_events FROM SplitInIcePulses;",
}

results = {}
for table, query in tqdm(tables_to_count.items(), desc="Zähle Events", unit="table"):
    results[table] = pd.read_sql(query, conn)["n_events"].iloc[0]

for table, n in results.items():
    print(f"Events in {table + ':':<25s} {n:>12,}")

In [ ]:
# Beispiel: 5 Events laden mit relevanten Größen für Muon Track Reco

# Pulse-Features (Input für das Netzwerk)
pulse_cols = "event_no, dom_x, dom_y, dom_z, dom_time, charge, hlc"

# Truth-Targets (das was rekonstruiert werden soll)
truth_cols = "event_no, azimuth, zenith, energy, position_x, position_y, position_z, track_length"

sample_events = pd.read_sql(
    f"SELECT DISTINCT event_no FROM truth LIMIT 5;", conn
)["event_no"].tolist()

placeholders = ",".join(str(e) for e in sample_events)

pulses = pd.read_sql(
    f"SELECT {pulse_cols} FROM SplitInIcePulses WHERE event_no IN ({placeholders});", conn
)
truth = pd.read_sql(
    f"SELECT {truth_cols} FROM truth WHERE event_no IN ({placeholders});", conn
)

print(f"Pulse-Daten: {len(pulses)} Pulse aus {pulses['event_no'].nunique()} Events")
print("--- Pulses (erste Zeilen) ---")
display(pulses.head(10))

print("\n--- Truth (Targets) ---")
display(truth)

In [ ]:
# Check: Sind die event_no in beiden Tabellen 1:1 gematcht?
events_only_in_truth = pd.read_sql("""
    SELECT COUNT(*) AS n FROM truth
    WHERE event_no NOT IN (SELECT DISTINCT event_no FROM SplitInIcePulses);
""", conn)["n"].iloc[0]

events_only_in_pulses = pd.read_sql("""
    SELECT COUNT(DISTINCT event_no) AS n FROM SplitInIcePulses
    WHERE event_no NOT IN (SELECT event_no FROM truth);
""", conn)["n"].iloc[0]

print(f"Events nur in truth (ohne Pulses):  {events_only_in_truth}")
print(f"Events nur in Pulses (ohne Truth):  {events_only_in_pulses}")

if events_only_in_truth == 0 and events_only_in_pulses == 0:
    print("\nAlles gematcht! Jedes Event hat sowohl Pulse als auch Truth-Info.")
else:
    print("\nACHTUNG: Nicht alle Events sind in beiden Tabellen vorhanden!")